In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [4]:
dataset_path = "/content/drive/MyDrive/Employee-Attrition-XAI/datasets/HR-Employee-Attrition.csv"
data = pd.read_csv(dataset_path)
print(data.shape)
data.info()

(1470, 35)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel     

In [5]:
data.drop(
    columns=[
        "EmployeeCount",
        "EmployeeNumber",
        "Over18",
        "StandardHours"
    ],
    inplace=True
)

print(data.shape)

(1470, 31)


In [6]:
data["Attrition"] = data["Attrition"].map({
    "No":0,
    "Yes":1
})

data["Attrition"].value_counts()



,count
Attrition,
0,1233
1,237


In [7]:
binary_mapping = {
    "Male":0,
    "Female":1,
    "No":0,
    "Yes":1
}

binary_columns = ["Gender","OverTime"]

for col in binary_columns:
    data[col] = data[col].map(binary_mapping)

data[["Gender","OverTime"]].head()

,Gender,OverTime
0,1,1
1,0,0
2,0,1
3,1,1
4,0,0


In [8]:
X = data.drop("Attrition", axis=1)

y = data["Attrition"]

print(X.shape)
print(y.shape)

(1470, 30)
(1470,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("Training:", X_train.shape)
print("Testing :", X_test.shape)
X_train.select_dtypes(include="object").columns

Training: (1176, 30)
Testing : (294, 30)


Index(['BusinessTravel', 'Department', 'EducationField', 'JobRole',
       'MaritalStatus'],
      dtype='object')

In [ ]:
data[["Gender", "OverTime", "Attrition"]].head()

,Gender,OverTime,Attrition
0,1,1,1
1,0,0,0
2,0,1,1
3,1,1,0
4,0,0,0


In [10]:
one_hot_columns = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "JobRole",
    "MaritalStatus"
]
preprocessor = ColumnTransformer(
    transformers=[
        (
            "onehot",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            one_hot_columns
        )
    ],
    remainder="passthrough"
)

In [11]:
X_train_encoded = preprocessor.fit_transform(X_train)

X_test_encoded = preprocessor.transform(X_test)
print("Before Encoding:", X_train.shape)

print("After Encoding :", X_train_encoded.shape)

Before Encoding: (1176, 30)
After Encoding : (1176, 44)


In [12]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)
scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train_encoded)

X_test_scaled = scaler.transform(X_test_encoded)

['onehot__BusinessTravel_Travel_Frequently'
 'onehot__BusinessTravel_Travel_Rarely'
 'onehot__Department_Research & Development' 'onehot__Department_Sales'
 'onehot__EducationField_Life Sciences' 'onehot__EducationField_Marketing'
 'onehot__EducationField_Medical' 'onehot__EducationField_Other'
 'onehot__EducationField_Technical Degree'
 'onehot__JobRole_Human Resources' 'onehot__JobRole_Laboratory Technician'
 'onehot__JobRole_Manager' 'onehot__JobRole_Manufacturing Director'
 'onehot__JobRole_Research Director' 'onehot__JobRole_Research Scientist'
 'onehot__JobRole_Sales Executive' 'onehot__JobRole_Sales Representative'
 'onehot__MaritalStatus_Married' 'onehot__MaritalStatus_Single'
 'remainder__Age' 'remainder__DailyRate' 'remainder__DistanceFromHome'
 'remainder__Education' 'remainder__EnvironmentSatisfaction'
 'remainder__Gender' 'remainder__HourlyRate' 'remainder__JobInvolvement'
 'remainder__JobLevel' 'remainder__JobSatisfaction'
 'remainder__MonthlyIncome' 'remainder__MonthlyRa

In [13]:
print("Training:", X_train_scaled.shape)

print("Testing :", X_test_scaled.shape)
import joblib

joblib.dump(preprocessor, "preprocessor.pkl")
joblib.dump(scaler, "scaler.pkl")

Training: (1176, 44)
Testing : (294, 44)


['scaler.pkl']

In [14]:
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train_scaled shape: (1176, 44)
X_test_scaled shape: (294, 44)
y_train shape: (1176,)
y_test shape: (294,)


In [17]:
import os

print(os.path.exists("/content/drive/MyDrive/Employee-Attrition-XAI/datasets/processed"))

True


In [18]:
import joblib

joblib.dump(
    X_train_scaled,
    "/content/drive/MyDrive/Employee-Attrition-XAI/datasets/processed/X_train_scaled.pkl"
)

joblib.dump(
    X_test_scaled,
    "/content/drive/MyDrive/Employee-Attrition-XAI/datasets/processed/X_test_scaled.pkl"
)

joblib.dump(
    y_train,
    "/content/drive/MyDrive/Employee-Attrition-XAI/datasets/processed/y_train.pkl"
)

joblib.dump(
    y_test,
    "/content/drive/MyDrive/Employee-Attrition-XAI/datasets/processed/y_test.pkl"
)

['/content/drive/MyDrive/Employee-Attrition-XAI/datasets/processed/y_test.pkl']

In [19]:
import joblib

joblib.dump(
    preprocessor,
    "/content/drive/MyDrive/Employee-Attrition-XAI/models/preprocessor.pkl"
)

joblib.dump(
    scaler,
    "/content/drive/MyDrive/Employee-Attrition-XAI/models/scaler.pkl"
)

['/content/drive/MyDrive/Employee-Attrition-XAI/models/scaler.pkl']